In [1]:
import pandas as pd
import numpy as np
import joblib

model    = joblib.load('../models/best_model.pkl')
scaler   = joblib.load('../models/scaler.pkl')
features = joblib.load('../models/feature_columns.pkl')

# Simulate exact high-risk customer
# Tenure=0, Complaint=Yes, Mobile, Days since order=20, low cashback
test_input = {
    'Tenure': 0,
    'PreferredLoginDevice': 1,   # Mobile
    'CityTier': 3,
    'WarehouseToHome': 30,
    'PreferredPaymentMode': 0,   # Cash on Delivery
    'Gender': 0,                  # Female
    'HourSpendOnApp': 1,
    'NumberOfDeviceRegistered': 2,
    'PreferredOrderCat': 3,       # Mobile
    'SatisfactionScore': 1,
    'MaritalStatus': 2,           # Single
    'NumberOfAddress': 2,
    'Complain': 1,                # Yes
    'OrderAmountHikeFromlastYear': 13,
    'CouponUsed': 0,
    'OrderCount': 1,
    'DaySinceLastOrder': 20,
    'CashbackAmount': 50.0,
}

df_test = pd.DataFrame([test_input])
df_test['EngagementScore'] = df_test['HourSpendOnApp']*0.4 + df_test['OrderCount']*0.4 + df_test['CouponUsed']*0.2
df_test['IsInactive']      = (df_test['DaySinceLastOrder'] >= 7).astype(int)
df_test['IsHighValue']     = (df_test['CashbackAmount'] >= 150).astype(int)
df_test['ComplainLowSat']  = ((df_test['Complain'] == 1) & (df_test['SatisfactionScore'] <= 2)).astype(int)

df_test = df_test[features]
scaled  = scaler.transform(df_test)

prob  = model.predict_proba(scaled)[0][1]
label = model.predict(scaled)[0]

print(f"Churn Probability: {prob*100:.1f}%")
print(f"Prediction: {'CHURN' if label==1 else 'STAY'}")

Churn Probability: 49.0%
Prediction: STAY
